# Lezione 8 — Gradienti: la direzione che migliora

**Come si usa questo notebook.** Come sempre. Prerequisito: Lezione 7 (vettori, matrici, e il prodotto scalare come somma pesata: e' il calcolo che i gradienti di oggi impareranno ad aggiustare).

**Cosa saprai fare alla fine:** capire *come* un modello impara — non per
magia, ma seguendo una derivata. È il concetto che rende comprensibile tutto
quello che Keras farà "da solo" nella Fase 2.

## Parte 1 — Teoria: la derivata è una sensibilità

Dimentica per un momento i limiti del liceo. Operativamente:

> **La derivata dice quanto cambia l'output se muovi l'input di un
> pochino.** `f'(x) = 3` significa: aumenti x di 0.001, f aumenta di circa
> 0.003. `f'(x) = -3`: f *diminuisce*.

Questa lettura basta per tutto il corso. E suggerisce subito un algoritmo:
se vuoi **minimizzare** f (per noi: l'errore del modello), guarda la
derivata e muoviti **contro** di essa — se la derivata è positiva, scendi a
sinistra; se è negativa, a destra.

Con più parametri, la derivata rispetto a ciascuno si impila in un vettore:
il **gradiente**. Il gradiente punta nella direzione di massima crescita;
il suo opposto, nella direzione di discesa più ripida. Da cui:

> **Discesa del gradiente**: `parametri = parametri - passo * gradiente`,
> ripetuto finché l'errore smette di scendere.

Il `passo` (learning rate) è il compromesso centrale: troppo piccolo e non
arrivi mai, troppo grande e scavalchi il minimo rimbalzando. Lo vedrai coi
tuoi occhi tra poco.

Ultima idea, la **chain rule**: se f è una catena di funzioni (e una rete
neurale lo è: strato dopo strato), la derivata della catena è il *prodotto*
delle derivate dei passi. È tutto ciò che serve per capire cosa fa la
"backpropagation": applica la chain rule all'indietro attraverso gli strati,
automaticamente. Keras la farà per te; ora sai *cosa* sta facendo.

In [ ]:
import numpy as np

def derivata_numerica(f, x, h=1e-6):
    return (f(x + h) - f(x - h)) / (2 * h)

def f(x):
    return x**2 + 3 * x              # derivata esatta: 2x + 3

for x in [-3.0, 0.0, 2.0]:
    print(f'x={x:+.0f}: numerica {derivata_numerica(f, x):+.4f}   esatta {2 * x + 3:+.4f}')

La derivata numerica — "muovi di poco e misura" — coincide con quella
esatta. È anche il tuo strumento di debug per sempre: qualsiasi gradiente
si può verificare così.

Ora guardiamo la discesa del gradiente muoversi, con tre learning rate:

In [ ]:
import matplotlib.pyplot as plt

def df(x):
    return 2 * x + 3

fig, assi = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
xs = np.linspace(-6, 3, 100)
for asse, passo in zip(assi, [0.05, 0.3, 1.05]):
    x, storia = 2.5, [2.5]
    for _ in range(15):
        x = x - passo * df(x)
        storia.append(x)
    asse.plot(xs, f(xs), lw=1)
    asse.plot(storia, [f(v) for v in storia], 'o-', ms=4)
    asse.set_title(f'passo = {passo}')
plt.suptitle('Discesa del gradiente: lento / giusto / troppo grande (diverge)')
plt.tight_layout()
plt.show()

Tre comportamenti che rivedrai per tutta la vita con i modelli veri:
a sinistra la discesa è corretta ma lenta; al centro arriva in pochi passi;
a destra il passo scavalca il minimo e **diverge**. Quando in Fase 2 un
training "esploderà", saprai qual è la prima manopola da girare.

## Parte 2 — Esercizio guidato

Il tuo compito, tutto nella cella sotto:

1. scrivi la funzione `g(x) = (x - 4)**2 + 1` e la sua derivata esatta;
2. verifica la derivata con `derivata_numerica` in x = 0 e x = 4;
3. esegui 20 passi di discesa del gradiente partendo da `x = -2` con
   passo 0.2, e controlla che il risultato finisca vicino a 4 (il minimo).

**Prova tu:**

In [ ]:
# Scrivi qui:
# 1) g e dg
# 2) verifica numerica in 0 e 4
# 3) 20 passi da x=-2, passo 0.2 -> x finale ~4

pass

### Soluzione spiegata

- la derivata di `(x-4)**2 + 1` è `2*(x-4)`: positiva a destra di 4,
  negativa a sinistra, **zero nel minimo** — nel minimo non c'è direzione
  che migliori, ed è per questo che la discesa si ferma lì;
- in x=4 la verifica numerica dà ~0: sei già nel punto piatto;
- 20 passi con passo 0.2 convergono comodamente.

In [ ]:
def g(x):
    return (x - 4)**2 + 1

def dg(x):
    return 2 * (x - 4)

for punto in [0.0, 4.0]:
    print(f'x={punto}: numerica {derivata_numerica(g, punto):+.4f}   esatta {dg(punto):+.4f}')

x = -2.0
for _ in range(20):
    x = x - 0.2 * dg(x)
print(f'\ndopo 20 passi: x = {x:.4f}  (minimo teorico: 4)')
assert abs(x - 4) < 0.05

## Parte 3 — Il progetto: Memory AI Lab, passo 8

Oggi il progetto **impara i suoi primi parametri**. Problema volutamente
semplice: prevedere `importance` di una memoria dalla lunghezza del testo,
con una retta `previsione = w * text_length + b`.

L'errore da minimizzare e' lo scarto quadratico medio: la media dei quadrati delle differenze tra previsione e valore vero (lo approfondiamo nella Lezione 9, insieme alle alternative per la classificazione). I gradienti di w e b si possono scrivere a mano - sono due derivate - e la discesa fa il resto. Nessuna libreria di ML: solo NumPy, il prodotto scalare come somma pesata della Lezione 7, e la teoria di oggi.

In [ ]:
import pandas as pd
from pathlib import Path

processed = Path('..') / 'datasets' / 'processed'
train = pd.read_csv(processed / 'memory_features_train.csv')
val = pd.read_csv(processed / 'memory_features_val.csv')

x_tr, t_tr = train['text_length'].to_numpy(), train['importance'].to_numpy()
x_va, t_va = val['text_length'].to_numpy(), val['importance'].to_numpy()

w, b, passo = 0.0, 0.0, 0.1
for epoca in range(200):
    previsione = w * x_tr + b
    errore = previsione - t_tr
    grad_w = 2 * (errore * x_tr).mean()     # derivata dell'errore rispetto a w
    grad_b = 2 * errore.mean()              # ... e rispetto a b
    w, b = w - passo * grad_w, b - passo * grad_b
    if epoca % 50 == 0:
        print(f'epoca {epoca:3d}: errore quadratico medio {np.mean(errore**2):.4f}')

mse_val = np.mean((w * x_va + b - t_va) ** 2)
mse_banale = np.mean((t_tr.mean() - t_va) ** 2)   # baseline: predire sempre la media
print(f'\nparametri imparati: w={w:+.3f}, b={b:+.3f}')
print(f'errore su validation: {mse_val:.4f}   baseline (media fissa): {mse_banale:.4f}')

Guarda l'errore scendere epoca dopo epoca: **questo è l'apprendimento**,
tutto qui — previsione, errore, gradiente, passo. Confronta poi l'errore di
validation con la baseline "predici sempre la media": se il modello la batte
anche di poco, ha estratto segnale vero; se non la batte, la feature non
basta — ed è un'informazione onesta, non un fallimento (la Fase 3, con gli
embedding del testo, darà feature molto più ricche).

## Cosa hai imparato

- La derivata è una **sensibilità**: quanto cambia l'output muovendo
  l'input. Il gradiente la estende a molti parametri.
- **Discesa del gradiente**: muoviti contro il gradiente, con un passo —
  troppo piccolo = lento, troppo grande = diverge.
- La derivata numerica verifica qualsiasi gradiente (debug per sempre).
- La **chain rule** moltiplica le derivate lungo la catena: la
  backpropagation è questo, applicato agli strati di una rete.
- Il ciclo previsione → errore → gradiente → aggiornamento è *tutto*
  l'apprendimento supervisionato.

## Quiz

1. Il gradiente di un parametro è -0.8. Per ridurre l'errore, il parametro
   va aumentato o diminuito?
2. Durante il training l'errore sale e scende in modo sempre più violento.
   Diagnosi e rimedio?
3. Nel progetto, perché il gradiente di w è `2 * (errore * x).mean()` e non
   solo `2 * errore.mean()`?

<details>
<summary><b>Apri le risposte</b></summary>

1. Aumentato: ci si muove contro il gradiente (`p = p - passo * (-0.8)`
   aumenta p). Il gradiente negativo dice che aumentando il parametro
   l'errore scende.
2. Learning rate troppo grande: la discesa scavalca il minimo e rimbalza
   (il terzo grafico della lezione). Rimedio: ridurre il passo.
3. Chain rule: l'errore dipende da w *attraverso* la previsione `w*x + b`,
   e la derivata della previsione rispetto a w è x. Il contributo di ogni
   esempio va quindi pesato per la sua x; b invece moltiplica 1.
</details>

## Fonti

- Goodfellow, Bengio, Courville, *Deep Learning*, cap. 4 (Numerical
  Computation: gradient-based optimization): https://www.deeplearningbook.org/
- NumPy, *the absolute basics* (operazioni usate qui):
  https://numpy.org/doc/stable/user/absolute_beginners.html

## Prossima lezione

Abbiamo minimizzato uno scarto quadratico "sulla fiducia". Ma *quale* errore
minimizzare — e perché per classificare serve la cross-entropy e non l'MSE —
è una scelta con conseguenze enormi. Lezione 9, l'ultima della Fase 0: alla
fine il progetto avrà un classificatore addestrato.